<a href="https://colab.research.google.com/github/astiakamila/Processing-Text/blob/main/102_(Velyn)_162_(Astia)_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> Tugas Pemteks: Feature Engineering </h1>
<h2> Astia Kamila Dwi Septiani </h2>
<h3> NIM: 25031554162 </h3>
<h2> Zikrah Neisya Fabiola Evelyna </h2>
<h3> Nim: 25031554102 </h3>

Instalasi Library

In [ ]:
!pip install gensim umap-learn plotly PySastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.2 MB/s eta 0:00:00


Import Library & Setup NLTK

In [ ]:
import warnings
from io import StringIO
import numpy as np
import pandas as pd
import requests
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.models
from umap import UMAP
import plotly.express as px

# Mengabaikan pesan warning
warnings.filterwarnings('ignore')

# Mengunduh resource NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Inisialisasi stemmer Sastrawi untuk Bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

Scraping & Preprocessing Data Cuaca

In [ ]:
# Mengambil data dari web BMKG
url_cuaca = "https://www.bmkg.go.id/cuaca/prakiraan-cuaca-indonesia.bmkg"
headers = {"User-Agent": "Mozilla/5.0"}
data_cuaca = []

try:
    res = requests.get(url_cuaca, headers=headers, timeout=10)
    if res.status_code == 200:
        soup = BeautifulSoup(res.text, "html.parser")
        elements = soup.find_all(['p', 'td', 'div'])
        for el in elements:
            txt = el.get_text(strip=True)
            if len(txt) > 15:
                data_cuaca.append(txt)
except Exception:
    pass

# Memakai data cadangan jika scraping tidak membuahkan hasil
if not data_cuaca:
    data_cuaca = [
        "Prakiraan cuaca wilayah Surabaya cerah berawan pada pagi hari dan berpotensi hujan ringan di sore hari.",
        "BMKG mengeluarkan peringatan dini cuaca ekstrem hujan lebat disertai angin kencang di sebagian wilayah Jawa Timur.",
        "Kondisi cuaca Jakarta diprakirakan cerah hingga berawan sepanjang hari ini.",
        "Suhu udara berkisar antara 24 hingga 33 derajat celsius dengan kelembapan tinggi.",
        "Hujan petir diprediksi terjadi di wilayah pegunungan pada malam hari nanti."
    ]

df_cuaca = pd.DataFrame({'teks_asli': data_cuaca})
sw_id = set(stopwords.words('indonesian'))

# Fungsi pembersihan teks Bahasa Indonesia (lowercase, hapus simbol, hapus stopword, stemming)
def preprocess_id(text):
    text = text.lower()
    text = nltk.re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = [w for w in text.split() if w not in sw_id]
    return " ".join([stemmer.stem(w) for w in tokens])

# Menerapkan preprocessing ke DataFrame cuaca
df_cuaca['prepro'] = df_cuaca['teks_asli'].apply(preprocess_id)
print("Data Cuaca setelah Preprocessing:")
print(df_cuaca.head())

Data Cuaca setelah Preprocessing:
                                           teks_asli  \
0  Prakiraan cuaca wilayah Surabaya cerah berawan...   
1  BMKG mengeluarkan peringatan dini cuaca ekstre...   
2  Kondisi cuaca Jakarta diprakirakan cerah hingg...   
3  Suhu udara berkisar antara 24 hingga 33 deraja...   
4  Hujan petir diprediksi terjadi di wilayah pegu...   

                                              prepro  
0  prakira cuaca wilayah surabaya cerah awan pagi...  
1  bmkg keluar ingat cuaca ekstrem hujan lebat se...  
2           kondisi cuaca jakarta prakira cerah awan  
3            suhu udara kisar derajat celsius lembap  
4          hujan petir prediksi wilayah gunung malam  


Scraping & Preprocessing Data Film

In [ ]:
# Mengambil data film dari Wikipedia
url_film = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
res_film = requests.get(url_film, headers=headers)
soup = BeautifulSoup(res_film.text, "html.parser")
table = soup.find("table", class_="wikitable")
df_film = pd.read_html(StringIO(str(table)))[0][['Title']].copy()

sw_en = set(stopwords.words('english'))

# Fungsi pembersihan teks Bahasa Inggris
def preprocess_en(text):
    text = str(text).replace('†', '').lower()
    text = nltk.re.sub(r'[^a-zA-Z\s]', '', text)
    return " ".join([w for w in text.split() if w not in sw_en])

# Menerapkan preprocessing ke DataFrame film
df_film['prepro'] = df_film['Title'].apply(preprocess_en)
print("Data Film setelah Preprocessing:")
print(df_film.head())

Data Film setelah Preprocessing:
                         Title                   prepro
0          Avengers: Endgame †         avengers endgame
1                       Avatar                   avatar
2  Spider-Man: Brand New Day †  spiderman brand new day
3     Avatar: The Way of Water         avatar way water
4                      Titanic                  titanic


Feature Engineering - Part 1 (Vectorization)

In [ ]:
print("--- CHAPTER 5: FEATURE ENGINEERING PART 1 ---")

# One-Hot Encoding pada data cuaca
vec_onehot = CountVectorizer(binary=True)
X_onehot_cuaca = vec_onehot.fit_transform(df_cuaca['prepro'])
df_onehot_cuaca = pd.DataFrame(X_onehot_cuaca.toarray(), columns=vec_onehot.get_feature_names_out())
print("\n[1] One Hot Encoding - Data Cuaca:")
print(df_onehot_cuaca.head())

# CountVectorizer Frekuensi Kata pada data cuaca
vec_count = CountVectorizer()
X_count_cuaca = vec_count.fit_transform(df_cuaca['prepro'])
df_count_cuaca = pd.DataFrame(X_count_cuaca.toarray(), columns=vec_count.get_feature_names_out())
print("\n[2] CountVectorizer (Frequency) - Data Cuaca:")
print(df_count_cuaca.head())

# TF-IDF Unigram & Bigram pada data cuaca
tfidf_ngram_cuaca = TfidfVectorizer(ngram_range=(1, 2))
X_ngram_cuaca = tfidf_ngram_cuaca.fit_transform(df_cuaca['prepro'])
df_tfidf_ngram_cuaca = pd.DataFrame(X_ngram_cuaca.toarray(), columns=tfidf_ngram_cuaca.get_feature_names_out())
print("\n[3] TF-IDF (Unigram + Bigram) - Data Cuaca:")
print(df_tfidf_ngram_cuaca.head())

# CountVectorizer Top 10 Kata pada data film
vec_count_film = CountVectorizer(max_features=10)
X_count_film = vec_count_film.fit_transform(df_film['prepro'])
df_count_film = pd.DataFrame(X_count_film.toarray(), columns=vec_count_film.get_feature_names_out())
print("\n[4] CountVectorizer (Top 10 Kata) - Data Film:")
print(df_count_film.head())

# TF-IDF Top 10 Fitur pada data film
vec_tfidf_film = TfidfVectorizer(max_features=10)
X_tfidf_film = vec_tfidf_film.fit_transform(df_film['prepro'])
df_tfidf_film = pd.DataFrame(X_tfidf_film.toarray(), columns=vec_tfidf_film.get_feature_names_out())
print("\n[5] TF-IDF (Top 10 Fitur) - Data Film:")
print(df_tfidf_film.head())

--- CHAPTER 5: FEATURE ENGINEERING PART 1 ---

[1] One Hot Encoding - Data Cuaca:
   angin  awan  bmkg  celsius  cerah  cuaca  derajat  ekstrem  gunung  hujan  \
0      0     1     0        0      1      1        0        0       0      1   
1      1     0     1        0      0      1        0        1       0      1   
2      0     1     0        0      1      1        0        0       0      0   
3      0     0     0        1      0      0        1        0       0      0   
4      0     0     0        0      0      0        0        0       1      1   

   ...  prakira  prediksi  ringan  serta  sore  suhu  surabaya  timur  udara  \
0  ...        1         0       1      0     1     0         1      0      0   
1  ...        0         0       0      1     0     0         0      1      0   
2  ...        1         0       0      0     0     0         0      0      0   
3  ...        0         0       0      0     0     1         0      0      1   
4  ...        0         1       0    

Analisis Rata-Rata Skor TF-IDF

In [ ]:
# Fungsi untuk menghitung rata-rata skor TF-IDF tertinggi
def get_top_tfidf_features(df_tfidf, n=5):
    mean_scores = df_tfidf.mean(axis=0).sort_values(ascending=False)
    return mean_scores.head(n)

print("--- ANALISIS KATA KUNCI UTAMA (RATA-RATA SKOR TF-IDF) ---")
print("\nTop 5 Kata/Frasa Utama - Data Cuaca:")
print(get_top_tfidf_features(df_tfidf_ngram_cuaca))

print("\nTop 5 Kata Utama - Data Film:")
print(get_top_tfidf_features(df_tfidf_film))

--- ANALISIS KATA KUNCI UTAMA (RATA-RATA SKOR TF-IDF) ---

Top 5 Kata/Frasa Utama - Data Cuaca:
cuaca         0.103855
hujan         0.101913
wilayah       0.101913
cerah awan    0.091717
awan          0.091717
dtype: float64

Top 5 Kata Utama - Data Film:
avengers     0.073099
avatar       0.060000
spiderman    0.060000
wars         0.053973
star         0.053973
dtype: float64


Feature Engineering - Part 2 (Word Embedding)

In [ ]:
print("--- CHAPTER 6: FEATURE ENGINEERING PART 2 ---")

# Tokenisasi kata untuk input Word Embedding
sentences_cuaca = [word_tokenize(doc.lower()) for doc in df_cuaca['prepro']]
sentences_film = [word_tokenize(doc.lower()) for doc in df_film['prepro']]

# Membuat dan menyimpan model Word2Vec untuk data cuaca
model_w2v = gensim.models.Word2Vec(sentences_cuaca, min_count=1, vector_size=100, window=5, sg=0)
model_w2v.save('cuaca.w2v')

# Fungsi untuk menghitung rerata vektor kata menjadi vektor dokumen
def get_doc_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(valid_vectors, axis=0) if valid_vectors else np.zeros(model.vector_size)

# Mengekstrak vektor dokumen cuaca dari Word2Vec
X_w2v_cuaca = np.array([get_doc_vector(doc, model_w2v) for doc in sentences_cuaca])
df_w2v_cuaca = pd.DataFrame(X_w2v_cuaca)
print("\n[1] Vektor Dokumen Word2Vec - Data Cuaca:")
print(df_w2v_cuaca.head())

# Membuat dan menyimpan model FastText untuk data film
model_fasttext = gensim.models.FastText(sentences_film, min_count=1, vector_size=100, window=5, min_n=3, max_n=6, sg=1, epochs=10)
model_fasttext.save('film.fasttext')

# Mengekstrak vektor dokumen film dari FastText
X_ft_film = np.array([get_doc_vector(doc, model_fasttext) for doc in sentences_film])
df_ft_film = pd.DataFrame(X_ft_film)
print("\n[2] Vektor Dokumen FastText - Data Film:")
print(df_ft_film.head())

--- CHAPTER 6: FEATURE ENGINEERING PART 2 ---

[1] Vektor Dokumen Word2Vec - Data Cuaca:
         0         1         2         3         4         5         6   \
0 -0.002244 -0.000615 -0.001187  0.000799  0.000354 -0.000201  0.000450   
1  0.000561  0.001365  0.000791  0.003380 -0.000455 -0.000355  0.001286   
2 -0.003485  0.002165 -0.001254 -0.003737  0.003434  0.000578  0.001761   
3  0.000989  0.004047  0.002370 -0.002867 -0.000583 -0.003490 -0.002535   
4 -0.001264 -0.000452  0.002435  0.006927 -0.000301 -0.002628  0.003935   

         7         8         9   ...        90        91        92        93  \
0  0.001903 -0.000553 -0.000831  ...  0.000331  0.002310  0.000773 -0.002399   
1  0.002205 -0.001604 -0.000791  ...  0.001448 -0.002098  0.000584 -0.001073   
2  0.004352 -0.002554 -0.000931  ...  0.002736  0.002880  0.003451 -0.001738   
3  0.003395  0.000344 -0.003310  ...  0.004493  0.001383 -0.000491 -0.000301   
4  0.001966 -0.004391 -0.002915  ... -0.001997  0.000795  0.

Visualisasi UMAP Embeddings

In [ ]:
# Visualisasi UMAP untuk Word2Vec (Data Cuaca)
if len(model_w2v.wv.index_to_key) > 1:
    umap_w2v = UMAP().fit_transform(model_w2v.wv.vectors)
    df_umap_w2v = pd.DataFrame(umap_w2v, columns=['umap1', 'umap2'])
    df_umap_w2v['word'] = model_w2v.wv.index_to_key

    fig_w2v = px.scatter(df_umap_w2v, x='umap1', y='umap2', text='word')
    fig_w2v.update_traces(textposition='top center')
    fig_w2v.update_layout(height=600, title_text='Word2Vec Visualization (Data Cuaca)')
    fig_w2v.show()

# Visualisasi UMAP untuk FastText (Data Film)
if len(model_fasttext.wv.index_to_key) > 1:
    umap_ft = UMAP().fit_transform(model_fasttext.wv.vectors)
    df_umap_ft = pd.DataFrame(umap_ft, columns=['umap1', 'umap2'])
    df_umap_ft['word'] = model_fasttext.wv.index_to_key

    fig_ft = px.scatter(df_umap_ft, x='umap1', y='umap2', text='word')
    fig_ft.update_traces(textposition='top center')
    fig_ft.update_layout(height=600, title_text='FastText Visualization (Data Film)')
    fig_ft.show()